# 3b 自定义GroupChatManager全部行为

In [7]:
import os
from typing import override
from rich import print
from dotenv import load_dotenv

In [8]:
load_dotenv("../.env/bailian", override=True)

# 这个例子需要结构化输出，但都是单值BooleanResult、StringResult这种，并不需要复杂的结构化，百炼接口可以胜任

True

In [9]:
model_id = os.environ["OPENAI_RESPONSES_MODEL_ID"]
model_id

'qwen3.5-plus'

In [10]:
from semantic_kernel.agents import (
    GroupChatOrchestration,
    ChatCompletionAgent,
    GroupChatManager,
    BooleanResult,
    StringResult,
    MessageResult,
)
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.connectors.ai.prompt_execution_settings import PromptExecutionSettings
from semantic_kernel.agents.runtime import InProcessRuntime

from semantic_kernel.contents import ChatHistory, ChatMessageContent, AuthorRole
from semantic_kernel.prompt_template import PromptTemplateConfig, KernelPromptTemplate
from semantic_kernel.kernel import Kernel, KernelArguments

from semantic_kernel.connectors.ai.chat_completion_client_base import ChatCompletionClientBase

In [11]:
farmer = ChatCompletionAgent(
    name="Farmer",
    description="A rural farmer from Southeast Asia.",
    instructions=(
        "You're a farmer from Southeast Asia. "
        "Your life is deeply connected to land and family. "
        "You value tradition and sustainability. "
        "You are in a debate. Feel free to challenge the other participants with respect."
    ),
    service=OpenAIChatCompletion(),
)
developer = ChatCompletionAgent(
    name="Developer",
    description="An urban software developer from the United States.",
    instructions=(
        "You're a software developer from the United States. "
        "Your life is fast-paced and technology-driven. "
        "You value innovation, freedom, and work-life balance. "
        "You are in a debate. Feel free to challenge the other participants with respect."
    ),
    service=OpenAIChatCompletion(),
)
teacher = ChatCompletionAgent(
    name="Teacher",
    description="A retired history teacher from Eastern Europe",
    instructions=(
        "You're a retired history teacher from Eastern Europe. "
        "You bring historical and philosophical perspectives to discussions. "
        "You value legacy, learning, and cultural continuity. "
        "You are in a debate. Feel free to challenge the other participants with respect."
    ),
    service=OpenAIChatCompletion(),
)
activist = ChatCompletionAgent(
    name="Activist",
    description="A young activist from South America.",
    instructions=(
        "You're a young activist from South America. "
        "You focus on social justice, environmental rights, and generational change. "
        "You are in a debate. Feel free to challenge the other participants with respect."
    ),
    service=OpenAIChatCompletion(),
)
spiritual_leader = ChatCompletionAgent(
    name="SpiritualLeader",
    description="A spiritual leader from the Middle East.",
    instructions=(
        "You're a spiritual leader from the Middle East. "
        "You provide insights grounded in religion, morality, and community service. "
        "You are in a debate. Feel free to challenge the other participants with respect."
    ),
    service=OpenAIChatCompletion(),
)
artist = ChatCompletionAgent(
    name="Artist",
    description="An artist from Africa.",
    instructions=(
        "You're an artist from Africa. "
        "You view life through creative expression, storytelling, and collective memory. "
        "You are in a debate. Feel free to challenge the other participants with respect."
    ),
    service=OpenAIChatCompletion(),
)
immigrant = ChatCompletionAgent(
    name="Immigrant",
    description="An immigrant entrepreneur from Asia living in Canada.",
    instructions=(
        "You're an immigrant entrepreneur from Asia living in Canada. "
        "You balance trandition with adaption. "
        "You focus on family success, risk, and opportunity. "
        "You are in a debate. Feel free to challenge the other participants with respect."
    ),
    service=OpenAIChatCompletion(),
)
doctor = ChatCompletionAgent(
    name="Doctor",
    description="A doctor from Scandinavia.",
    instructions=(
        "You're a doctor from Scandinavia. "
        "Your perspective is shaped by public health, equity, and structured societal support. "
        "You are in a debate. Feel free to challenge the other participants with respect."
    ),
    service=OpenAIChatCompletion(),
)

In [12]:
agents = [
    farmer,
    developer,
    teacher,
    activist,
    spiritual_leader,
    artist,
    immigrant,
    doctor,
]

In [ ]:
class ChatCompletionGroupChatManager(GroupChatManager):
    """A simple chat completion base group chat manager.

    This chat completion service requires a model that supports structured output.
    """

    service: ChatCompletionClientBase

    topic: str

    termination_prompt: str = (
        "You are mediator that guides a discussion on the topic of '{{$topic}}'. "
        "You need to determine if the discussion has reached a conclusion. "
        "If you would like to end the discussion, please respond with True. Otherwise, respond with False."
    )

    selection_prompt: str = (
        "You are mediator that guides a discussion on the topic of '{{$topic}}'. "
        "You need to select the next participant to speak. "
        "Here are the names and descriptions of the participants: "
        "{{$participants}}\n"
        "Respond with only the name of the participant you would like to select."
    )

    result_filter_prompt: str = (
        "You are mediator that guides a discussion on the topic of '{{$topic}}'. "
        "You have just concluded the discussion. "
        "Please summarize the discussion and provide a closing statement."
    )

    def __init__(self, topic: str, service: ChatCompletionClientBase, **kwargs) -> None:
        """Initialize the group chat manager."""
        super().__init__(topic=topic, service=service, **kwargs)

    async def _render_prompt(self, prompt: str, arguments: KernelArguments) -> str:
        """填充提示词模板中的变量"""
        prompt_template_config = PromptTemplateConfig(template=prompt)
        prompt_template = KernelPromptTemplate(prompt_template_config=prompt_template_config)
        return await prompt_template.render(Kernel(), arguments=arguments)

    @override
    async def should_request_user_input(self, chat_history: ChatHistory) -> BooleanResult:
        """要不要用户介入"""
        return BooleanResult(
            result=False,
            reason="This group chat manager does not require user input.",
        )

    @override
    async def should_terminate(self, chat_history: ChatHistory) -> BooleanResult:
        """对话结束条件"""
        should_terminate = await super().should_terminate(chat_history)
        if should_terminate.result:
            return should_terminate

        chat_history.messages.insert(
            0,
            ChatMessageContent(
                role=AuthorRole.SYSTEM,
                content=await self._render_prompt(
                    self.termination_prompt,
                    KernelArguments(topic=self.topic),
                ),
            ),
        )
        chat_history.add_message(
            ChatMessageContent(role=AuthorRole.USER, content="Determine if the discussion should end."),
        )

        response = await self.service.get_chat_message_content(
            chat_history,
            settings=PromptExecutionSettings(response_format=BooleanResult),
        )

        bool_with_reason = BooleanResult.model_validate_json(response.content)

        print(f"[#7F8C8D]Should terminate: {bool_with_reason.result}\nReason: {bool_with_reason.reason}.[/#7F8C8D]")

        return bool_with_reason

    @override
    async def select_next_agent(
        self,
        chat_history: ChatHistory,
        participant_descriptions: dict[str, str],
    ) -> StringResult:
        """确定下一个发言的人"""
        # 填充prompt模板占位符
        selection_prompt_filled = await self._render_prompt(
            self.selection_prompt,
            KernelArguments(
                topic=self.topic,
                participants="\n".join([f"{k}: {v}" for k, v in participant_descriptions.items()]),
            ),
        )
        # 插入selection_prompt提示词
        chat_history.messages.insert(
            0,
            ChatMessageContent(
                role=AuthorRole.SYSTEM,
                content=selection_prompt_filled,
            ),
        )
        # 插入用户指令
        chat_history.add_message(
            ChatMessageContent(
                role=AuthorRole.USER,
                content="Now select the next participant to speak.",
            ),
        )

        # 调用模型获取下一个发言人
        response = await self.service.get_chat_message_content(
            chat_history,
            settings=PromptExecutionSettings(response_format=StringResult),
        )
        name_with_reason = StringResult.model_validate_json(response.content)

        print(f"[#7F8C8D]Next participant: {name_with_reason.result}Reason: {name_with_reason.reason}.[/#7F8C8D]")

        if name_with_reason.result in participant_descriptions:
            return name_with_reason

        raise RuntimeError(f"Unknown participant selected: {response.content}.")

    @override
    async def filter_results(self, chat_history: ChatHistory) -> MessageResult:
        """过滤后返回MessageResult"""
        if not chat_history.messages:
            raise RuntimeError("No messages in the chat history.")

        # 填充prompt占位符
        result_filter_prompt_filled = await self._render_prompt(
            self.result_filter_prompt,
            KernelArguments(topic=self.topic),
        )
        # 插入result_filter_prompt
        chat_history.messages.insert(
            0,
            ChatMessageContent(
                role=AuthorRole.SYSTEM,
                content=result_filter_prompt_filled,
            ),
        )
        # 插入用户指令
        chat_history.add_message(
            ChatMessageContent(role=AuthorRole.USER, content="Please summarize the discussion."),
        )

        # 调用模型总结内容
        response = await self.service.get_chat_message_content(
            chat_history,
            settings=PromptExecutionSettings(response_format=StringResult),
        )
        str_with_reason = StringResult.model_validate_json(response.content)

        return MessageResult(
            result=ChatMessageContent(role=AuthorRole.ASSISTANT, content=str_with_reason.result),
            reason=str_with_reason.reason,
        )


In [14]:
def agent_response_callback(message: ChatMessageContent) -> None:
    """Observer function to print the messages from the agents."""
    name = message.name.lower()
    if "farmer" in name:
        print(f"[#E74C3C]{message.name} ▶︎ {message.content}[/#E74C3C]")
    elif "developer" in name:
        print(f"[#2ECC71]{message.name} ▶︎ {message.content}[/#2ECC71]")
    elif "teacher" in name:
        print(f"[#3498DB]{message.name} ▶︎ {message.content}[/#3498DB]")
    elif "activist" in name:
        print(f"[#2ECC7A]{message.name} ▶︎ {message.content}[/#2ECC7A]")
    elif "spiritual" in name:
        print(f"[#E67E22]{message.name} ▶︎ {message.content}[/#E67E22]")
    elif "artist" in name:
        print(f"[#F1C40F]{message.name} ▶︎ {message.content}[/#F1C40F]")
    elif "immigrant" in name:
        print(f"[#9B59B6]{message.name} ▶︎ {message.content}[/#9B59B6]")
    else:
        print(f"{message.name} ▶︎ {message.content}")

In [15]:
gcor = GroupChatOrchestration(
    members=agents,
    manager=ChatCompletionGroupChatManager(
        topic="What does a good life mean to you personally?",
        service=OpenAIChatCompletion(),
        max_rounds=10,
    ),
    agent_response_callback=agent_response_callback,
)

In [16]:
runtime = InProcessRuntime()
runtime.start()

gcor_result = await gcor.invoke(
    task="Please start the discussion.",
    runtime=runtime,
)
value = await gcor_result.get()

Should terminate: False
Reason: The discussion has just begun, and no participants have shared their perspectives yet. There is no 
conclusion reached on the topic of 'What does a good life mean to you personally?'. Further dialogue is needed..

Next participant: FarmerReason: The Farmer offers a grounded, rural perspective that contrasts with urban and 
technological viewpoints, providing a foundational understanding of a good life rooted in nature, simplicity, and 
community—ideal for opening the discussion..

Farmer ▶︎ Ah, good morning, friends. I’m sitting here under the shade of a big mango tree on my farm in northern 
Thailand, sipping some fresh lemongrass tea. The rice paddies are turning golden—soon it will be harvest time. For 
generations, my family has farmed this land, planting rice the way our ancestors taught us: with care, with rhythm,
and always listening to the seasons.

But lately… things feel different. The rains come late or too strong. The soil doesn’t hold water like it used to. 
And I see young people leaving the villages for cities, chasing jobs that pull them away from the earth.

I’ve been thinking—how can we honor tradition while also adapting? Some say we must go fully modern: new seeds, 
chemicals, machines. But I worry—if we lose our old ways, what happens to the balance? To the knowledge passed down
like seeds from hand to hand?

So I ask you: In your experience, how do we grow food *and* keep our roots? Can progress and tradition walk side by
side—or must one give way?

Should terminate: False
Reason: The discussion has just begun. The farmer has introduced a thoughtful question about balancing tradition 
and progress in agriculture, but no other participants have responded yet. There is no conclusion or resolution 
reached on the broader topic of 'What does a good life mean to you personally?' The exchange needs more dialogue to
develop..

Next participant: DeveloperReason: The Farmer has raised important questions about tradition, sustainability, and 
the balance between old and new ways of living. The Developer, coming from a contrasting urban and technological 
background, can offer a different perspective on progress and modernity, potentially creating a meaningful dialogue
on how different lifestyles define a good life..

Developer ▶︎ Appreciate you painting that picture, my friend. I can almost feel the humidity in the air and hear the
rustle of rice stalks in the wind. From where I sit—on a rooftop terrace in Austin, Texas, with drones buzzing 
overhead and electric scooters zipping down the street below—I realize how different our worlds seem. But your 
question? That’s universal.

You’re asking whether progress has to mean erasure. And I’ll be honest: as someone who builds software for a 
living, I’ve seen innovation move fast—sometimes too fast. We optimize for speed, efficiency, scalability… but we 
often forget to ask: *What are we losing in the process?*

So let me flip it back to you with a story. A few years ago, I worked on an agri-tech startup—an app to help small 
farmers track soil health using satellite data and AI. Cool tech, right? But when we piloted it in rural Kansas, 
many older farmers just shrugged. One guy said, “Son, I’ve been reading the sky and the soil since before your 
daddy was born. Your app tells me what I already know—and sometimes gets it wrong.”

That hit me. Not because tech is useless—but because *imposition* is dangerous. Progress shouldn’t override 
tradition; it should *amplify* it.

Imagine if your ancestors had access to real-time weather modeling or drought-resistant seed banks developed 
through open-source science—*while still planting by the moon and honoring seasonal rhythms*. What if we used 
drones not to replace labor, but to map water runoff so your community could restore ancient irrigation techniques 
more effectively?

Tradition holds wisdom. But climate change? That’s new. The monsoon shifts you’re seeing—it’s not just a variation.
It’s a signal. And while your ancestors adapted over centuries, we now have to adapt within decades. So maybe the 
answer isn’t *machine vs. hand*, or *chemical vs. natural*—but *how do we blend them with respect?*

I think progress and tradition *can* walk side by side—but only if progress listens first. Not as a savior, but as 
a partner.

So here’s my challenge back to you—and to all of us: How do we build tools that don’t erase your lemongrass tea 
moments, but protect them? How do we innovate *with* the land, not just *on* it?

And seriously—would your community be open to testing low-cost sensors to track soil moisture? Paired with your 
ancestral knowledge, that data could be powerful. Just a thought.

Should terminate: False
Reason: The discussion has evolved meaningfully between two distinct perspectives—Farmer and Developer—touching on 
tradition, progress, environmental change, and the integration of knowledge systems. Both participants have shared 
personal reflections, posed thoughtful questions, and offered potential collaboration, indicating a mutual 
understanding and openness to continued dialogue. However, the core question of 'What does a good life mean to you 
personally?' has not yet been directly addressed by either party in a way that centers personal values, 
fulfillment, or inner definition of a good life. The conversation remains focused on external practices (farming, 
technology) rather than internal meaning. To reach a true conclusion, deeper personal reflection on the original 
question is still needed..

Next participant: TeacherReason: The Developer raised important questions about technology and tradition, and the 
Teacher—having witnessed historical shifts in values, education, and society—can offer a reflective, long-term 
perspective on how cultures preserve meaning while adapting to change..

Teacher ▶︎ Ah, my friend from Austin with the drones and the rooftop terrace—thank you. Your story about the farmer
in Kansas… *da*, I know that man. He could be my cousin in Ukraine or my neighbor here in Thailand. His words are 
not resistance to change—they are a quiet defense of dignity, of lived truth.

You speak of partnership between old and new, and I agree: progress must not come like an army, but like a guest. 
And every wise host knows—guests do not dictate the meal.

Let me tell you something from my own past. I was born in 1947, in a village near Lviv, when it was still part of 
the Soviet Union. The state sent "agricultural experts" with tractors and chemical fertilizers, promising 
abundance. They said our horse-drawn plows and crop rotations were backward. “Science will feed you now,” they 
declared.

And for a time—yes, yields rose. But by the 1970s? The soil turned gray. The rivers lost their fish. The old seed 
varieties—the ones that survived drought, flood, even war—were replaced, then forgotten. When the system collapsed,
we had no resilience left. Not in the economy, and not in the earth.

So I say this carefully: **technology is neutral, but power is not.** A drone can map water runoff… or it can 
enforce land grabs. A sensor can preserve tradition… or it can become a tool for corporations to extract data while
selling us expensive inputs we never needed before.

This is why your question—"Would your community accept sensors?"—must be followed by deeper ones:  
Who owns the data?  
Who decides how it’s used?  
And if one day the satellite goes dark—will the young people still know how to read the clouds?

I am not against knowledge—*Bog s’vidit*, no! I taught history for forty years. I believe in learning. But I also 
believe in *roots as archives*. Our rice-planting songs here in Thailand? They are not just melodies—they contain 
instructions. The rhythm matches the pace of transplanting. The lyrics name the stars that appear when it’s time to
sow. This is science, too—just not written in code.

So let us imagine another kind of innovation:  
What if we recorded these songs with young farmers, pairing them with simple apps that play the right verse at the 
right season—like a digital elder whispering in your ear?  
What if open-source cooperatives—not Silicon Valley startups—developed low-cost sensors, shared freely among 
villages, built by local hands and repaired without permission?

Ah, you smile—I see you understand.

Progress is not evil. But *extractive* progress—the kind that takes wisdom, packages it, sells it back as a 
subscription—is a thief in clean shoes.

Now let me ask *you*, builder of software:  
Can you imagine a world where technology serves memory instead of replacing it?  
Where your AI doesn’t predict planting dates—but learns from generations of farmers, and then *defers* to them?

And to answer your challenge: Yes, perhaps my neighbors would try soil sensors. But only if they are taught how to 
build them, not just use them. Only if the tools remain theirs—like the mango tree under which I sit, whose roots 
go deep, but whose branches reach for light.

Because tradition without adaptation becomes ritual without meaning.  
And innovation without humility becomes noise without soul.

So let us walk side by side, yes—but let us agree on the path first. Not the one paved with venture capital, but 
the old footpath through the rice fields, worn down by bare feet and centuries of care.

That, my friend, is where the future should begin.

Should terminate: True
Reason: The discussion has evolved from personal reflections on a good life—rooted in tradition, dignity, and 
harmony with the land—to a shared vision of balanced progress that honors ancestral wisdom while thoughtfully 
integrating technology. The Farmer, Developer, and Teacher have each contributed meaningful perspectives, building 
mutual understanding and arriving at a nuanced synthesis: innovation must serve, not supplant, lived knowledge. 
While the conversation could continue, it has reached a natural and thoughtful conclusion grounded in respect, 
reciprocity, and hope..

In [18]:
await runtime.stop_when_idle()